# Week 2 — Deep Learning 입문 + mini UNet 학습

## 이번 주 학습 목표
1. **W1 baseline** 의 한계를 명확히 인지 (k가 클수록 |Δφ| 악화)
2. **2D UNet** 아키텍처 핵심 (encoder/decoder + skip-connection) 직접 분석
3. **SliceDataset** 으로 슬라이스를 양옆 이웃에서 예측하는 학습 데이터 구성
4. **mini UNet** (~30K~120K params) 을 학생 노트북 CPU에서 직접 학습
   - **PRESET 옵션** — `'fast'` (10분), `'standard'` (30분), `'full'` (60분)
5. 학습된 모델의 |Δφ|·SSIM 을 **W1 Linear baseline 과 직접 비교**

## 노트북 사용 방법

본 노트북의 모델과 학습 루프는 helpers/model_utils.py에 정의되어 있습니다. 본문에서는 preset과 파라미터를 바꾸어가며 학습 곡선과 평가 지표 변화를 분석합니다.

본문 **[Try-it!] · §6.5 심화** 블록에서 preset · k · lr · loss 등을 sweep하면서 학습 곡선과 평가 지표의 변화를 관찰. 정답 박스는 없습니다. 본인 노트에 가설·관찰·분석을 자유롭게 기록.

본 노트북은 학생 노트북(CPU)에서 동작합니다. GPU 있으면 자동 사용.

## 0. 환경 준비

In [ ]:
import sys
from pathlib import Path

# helpers(dr_utils.py · model_utils.py)는 같은 폴더(다운로드 zip) 또는 ../helpers(저장소)에 있을 수 있음
for _cand in [Path('.'), Path('..') / 'helpers']:
    if (_cand / 'dr_utils.py').exists():
        sys.path.insert(0, str(_cand.resolve())); break

import numpy as np
import matplotlib.pyplot as plt
import torch

from dr_utils import (
    load_volume, porosity, predict_linear_k, eval_targets,
    porosity_error, ssim_3d_mean,
    setup_plot_style, ORANGE, NAVY, GREEN, RED, GRAY,
)
from model_utils import (
    UNetMini, count_parameters, SliceDataset,
    train_quick, evaluate_model,
    save_ckpt, load_ckpt, TRAINING_PRESETS,
)
setup_plot_style()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}, device={DEVICE}')

## 1. W1 baseline 복습 — 우리가 이길 대상



본 W2의 모든 결과는 "W1 Linear baseline 대비 얼마나 좋아졌나" 로 평가합니다.

In [ ]:
# data/ 는 같은 폴더(다운로드) 또는 ../data(저장소)에 있을 수 있음
DATA = next((p for p in [Path('data'), Path('..') / 'data'] if (p / 'BB_256.bin').exists()), Path('data'))
bb = load_volume(DATA / 'BB_256.bin')
print('data dir:', DATA)

# W1 B1 Linear baseline — 우리가 이길 대상. 이웃 거리 k=1 (각 슬라이스를 양옆에서 예측)
K = 1
rec_l = predict_linear_k(bb, K)
m_baseline = eval_targets(rec_l, bb, K)
print(f'B1 Linear (k={K})   |Δφ|={m_baseline["dphi_pp"]:.2f}%p   SSIM={m_baseline["ssim"]:.4f}')

## 2. Mini UNet 아키텍처 분석



UNet 구조:

- **Encoder** (downsampling) — Conv → Pool 반복으로 추상 특징 추출

- **Decoder** (upsampling) — ConvTranspose 로 해상도 복원

- **Skip-connection** — Encoder의 detail을 Decoder에 직접 전달 (UNet의 핵심!)



본 mini UNet:

- 입력: 2-channel `(slice_before, slice_after)` ← W1 linear 보간이 쓰던 "이웃 슬라이스" 2장

- 출력: 1-channel `(slice_middle)` — 가운데 슬라이스 예측 (sigmoid → [0, 1])

- 3 단계 encoder/decoder (depth 3)

In [ ]:
# 세 preset의 모델 크기 비교

print(f"{'preset':<10}{'base':>5}{'params':>10}")

for name, p in TRAINING_PRESETS.items():

    m = UNetMini(in_ch=2, base=p['base'])

    print(f'{name:<10}{p["base"]:>5}{count_parameters(m):>10,}')



# 모델 구조 출력

print('\n--- UNetMini(base=16) 구조 ---')

print(UNetMini(in_ch=2, base=16))

## 3. SliceDataset — 학습 데이터 (이웃 거리 k)

각 sample: `(input_2ch = [vol[t−k], vol[t+k]],  target_1ch = vol[t])`
즉 슬라이스 t 를 양옆 이웃 `t−k`·`t+k` 에서 예측하도록 학습합니다.

- k=1 → 바로 옆 이웃에서 예측 (가장 쉬움). k 가 커질수록 어려워짐
- patch_size=64 → 256×256 에서 64×64 random crop (학습 가속)
- augment=True → flip 증강

In [ ]:
ds = SliceDataset(bb, k=K, patch_size=64, n_patches_per_triplet=4, augment=True)

print(f'Dataset size: {len(ds)} samples')



# 한 sample 시각화

x, y = ds[0]

print(f'x.shape={x.shape}  y.shape={y.shape}')



fig, axes = plt.subplots(1, 3, figsize=(10, 4))

axes[0].imshow(x[0]); axes[0].set_title('입력 이웃 t−k'); axes[0].axis('off')

axes[1].imshow(x[1]); axes[1].set_title('입력 이웃 t+k'); axes[1].axis('off')

axes[2].imshow(y[0]); axes[2].set_title('예측 대상 t (GT)'); axes[2].axis('off')

plt.tight_layout(); plt.show()

## 4. 학습 — PRESET 선택!



본인 노트북 사양/시간에 맞춰 선택:



| PRESET | base | epochs | 예상 시간 (CPU) | 결과 품질 |

|---|---|---|---|---|

| `'fast'`     | 8  | 20  | ~10 분 | 작동 확인 |

| `'standard'` | 16 | 50  | ~30 분 | 의미있는 비교 |

| `'full'`     | 16 | 100 | ~60 분 | baseline 명확히 능가 |

In [ ]:
PRESET = 'fast'  # ← 본인 환경에 맞춰 변경. 'standard' / 'full' 도 가능.



model, history = train_quick(bb, k=K, preset=PRESET, device=DEVICE, verbose=True)

In [ ]:
# 학습 곡선

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(history, color=ORANGE, lw=2)

ax.set_xlabel('Epoch'); ax.set_ylabel('Train L1 loss')

ax.set_title(f'mini UNet 학습 곡선 (preset={PRESET})')

ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()



# 체크포인트 저장 (나중에 다시 학습 안 해도 됨)

save_ckpt(model, f'unet_mini_{PRESET}.pth',

          meta={'base': TRAINING_PRESETS[PRESET]['base'], 'preset': PRESET, 'k': K})

print(f'✓ saved unet_mini_{PRESET}.pth')

## 5. 평가 — W1 Linear vs UNet mini

각 슬라이스 t 를 모델로 예측해 |Δφ|·SSIM 을 linear baseline 과 비교합니다.

> 이웃 거리 **k=1** (바로 옆에서 예측) 에서 UNet 과 linear 를 겨뤄 봅니다. k 가 커지면(이웃이 멀어지면) 누가 더 강한지는 §6.5-C 에서 직접 확인하세요.

In [ ]:
res_unet = evaluate_model(model, bb, k=K, device=DEVICE)
print(f'B1 Linear     |Δφ|={m_baseline["dphi_pp"]:.2f}%p   SSIM={m_baseline["ssim"]:.4f}')
print(f'UNet ({PRESET}) |Δφ|={res_unet["dphi_pp"]:.2f}%p   SSIM={res_unet["ssim"]:.4f}')

# 해석: UNet 은 pore 구조를 학습하므로 구조(SSIM)에서 강점을 가집니다.
#   이웃 거리 k 가 커질수록(예측이 멀어질수록) 누가 더 강한지는 §6.5-C 에서 직접 확인하세요.
verdict = 'UNet 우세' if res_unet['ssim'] > m_baseline['ssim'] else 'linear 우세 → §6.5 로 개선 탐구'
print(f'\nSSIM 기준 판정: {verdict}')

In [ ]:
# 시각: 한 슬라이스에서 GT vs B1 vs UNet 비교
z_show = 62
recon_unet = res_unet['recon']
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(bb[z_show]); axes[0].set_title(f'GT z={z_show}')
axes[1].imshow(rec_l[z_show]); axes[1].set_title('B1 Linear')
axes[2].imshow(recon_unet[z_show]); axes[2].set_title(f'UNet ({PRESET})')
diff = np.abs(bb[z_show].astype(float) - recon_unet[z_show])
axes[3].imshow(diff, cmap='hot'); axes[3].set_title('|GT − UNet|  (밝을수록 오차)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 6. 박스 모음



### PRESET 변경 → 학습 시간 vs 정확도

위 4번 셀에서 `PRESET = 'standard'` 또는 `'full'` 로 바꿔 재학습.

**비교 표 작성:** preset별 시간 / 파라미터 / |Δφ| / SSIM 4-row.



### sparse k 변경

본 노트북은 K=5 학습. K=3 또는 K=7로 바꿔 학습/평가.

- K=3 (쉬움) → UNet의 |Δφ| 가 baseline 대비 얼마나 작아지나?

- K=7 (어려움) → UNet도 baseline 정도로 떨어지지 않나?



### 다른 도메인에서 평가

BB에서 학습한 모델을 CastleGate, Bentheimer, Parker 에 평가 (zero-shot 비슷).

도메인 generalization 어느 정도?

In [ ]:
# BB 학습 모델을 다른 도메인에 평가

for name in ['CastleGate', 'Bentheimer', 'Parker']:

    vol = load_volume(DATA / f'{name}_256.bin')

    res = evaluate_model(model, vol, k=K, device=DEVICE)

    print(f'  {name:12s}  UNet |Δφ|={res["dphi_pp"]:.2f}%p  SSIM={res["ssim"]:.4f}')

## 6.5 심화 — 직접 바꾸고 분석하기 (Advanced)

지금까지는 `train_quick` 같은 편의 함수가 학습 루프를 감춰 줬습니다. 아래에서는 그 루프를 **직접 펼쳐**
손실·학습률·구조를 자유롭게 바꿉니다. 정답 코드를 그대로 돌리기보다, **인자/구조/손실을 직접 바꾸고 그 영향을
정량적으로 분석**하는 것이 목표입니다. 각 셀의 `# TODO`를 채우고 관찰을 한두 문장으로 적으세요.

In [ ]:
# === A. 학습 루프를 직접 제어하기 (train_quick을 펼친 형태) ===
import torch.nn as nn
from torch.utils.data import DataLoader
from model_utils import train_one_epoch

def train_custom(volume, k=1, base=8, epochs=20, lr=1e-3, batch=4,
                 patch=64, criterion=None, device=DEVICE, verbose=False):
    """train_quick을 펼친 버전 — 손실/lr/구조를 직접 바꿔 실험하세요."""
    criterion = criterion if criterion is not None else nn.L1Loss()
    ds = SliceDataset(volume, k=k, patch_size=patch, n_patches_per_triplet=4, augment=True)
    loader = DataLoader(ds, batch_size=batch, shuffle=True)
    model = UNetMini(in_ch=2, base=base).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = []
    for ep in range(epochs):
        loss = train_one_epoch(model, loader, opt, criterion, device=device)
        hist.append(loss)
        if verbose and (ep + 1) % 5 == 0:
            print(f'  ep{ep+1:3d}  loss={loss:.4f}')
    return model, hist

m_custom, h_custom = train_custom(bb, k=K, base=8, epochs=20, lr=1e-3, verbose=True)
print('최종 loss:', round(h_custom[-1], 4))

In [ ]:
# === B. 손실 함수 비교 — L1 vs MSE (그리고 직접 더 추가) ===
results_loss = {}
for name, crit in [('L1', nn.L1Loss()), ('MSE', nn.MSELoss())]:
    m, h = train_custom(bb, k=K, base=8, epochs=20, criterion=crit)
    r = evaluate_model(m, bb, k=K, device=DEVICE)
    results_loss[name] = (h, r)
    print(f'{name:4s}  |Δφ|={r["dphi_pp"]:.2f}%p  SSIM={r["ssim"]:.4f}')

# 학습 곡선 겹쳐 그리기
plt.figure(figsize=(7, 4))
for name, (h, _) in results_loss.items():
    plt.plot(h, lw=2, label=name)
plt.xlabel('epoch'); plt.ylabel('train loss'); plt.legend(); plt.grid(alpha=0.3)
plt.title('L1 vs MSE 학습 곡선'); plt.tight_layout(); plt.show()

# TODO: 어느 손실이 더 선명/안정적인가? |Δφ|·SSIM과 곡선 모양으로 근거를 적으세요.
# TODO(도전): L1 + λ·(1−SSIM) 복합 손실을 직접 구현해 비교 (힌트: dr_utils.ssim_2d 활용).

In [ ]:
# === C. k-regime crossover — UNet은 어느 k에서 linear를 이기나? ===
# 각 이웃 거리 k 에서 '따로 학습' 하고 같은 k 의 linear 와 비교
print(f"{'k':>2} | linear SSIM | UNet SSIM | 승자")
for k_test in [1, 2, 3, 5]:
    lin = eval_targets(predict_linear_k(bb, k_test), bb, k_test)
    mk, _ = train_custom(bb, k=k_test, base=8, epochs=20)
    rk = evaluate_model(mk, bb, k=k_test, device=DEVICE)
    print(f"{k_test:>2} | {lin['ssim']:>11.4f} | {rk['ssim']:>9.4f} | {'UNet' if rk['ssim']>lin['ssim'] else 'linear'}")
# TODO: k 가 커질수록(이웃이 멀어질수록) UNet 과 linear 중 누가 유리한지 관찰하고 이유를 적어보세요.

In [ ]:
# === D. per-slice 오차 — 모델은 '어디서' 실패하나? ===
res5 = evaluate_model(model, bb, k=K, device=DEVICE)   # §5에서 학습한 model 재사용
recon = res5['recon']
per_slice = np.array([abs(recon[z].mean() - bb[z].mean()) for z in range(bb.shape[0])])

plt.figure(figsize=(9, 3))
plt.plot(per_slice * 100, color=RED, lw=1.8)
plt.xlabel('slice z'); plt.ylabel('|Δφ| (%p)'); plt.title('per-slice 오차')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

worst = per_slice.argsort()[-5:][::-1]
print('오차 가장 큰 슬라이스 z =', worst.tolist())
# TODO: worst 슬라이스를 plt.imshow(bb[z])로 띄워, 어떤 구조에서 실패하는지 관찰·서술.

### 🏁 챌린지 — 4 도메인 모두에서 Linear를 이겨라

`train_custom`의 인자(base · epochs · lr · k · 손실)를 자유롭게 조합해,
**BB · CastleGate · Bentheimer · Parker 네 도메인 모두에서** B1 Linear 베이스라인보다
|Δφ| 또는 SSIM을 개선하는 설정을 찾아보세요. 최종 설정과 근거를 정리합니다.
쉽지 않습니다 — *학습 도메인 적합 ↔ cross-domain 일반화 ↔ 과적합* 사이의 균형이 관건입니다.

## 7. 다음 주 (W3) — 적대적 학습 (pix2pix GAN)

- 본 W2는 L1 loss만 사용. W3에서 **Discriminator(PatchGAN)** 를 더해 더 선명한 복원으로 확장합니다.
- 준비물: 본 주차 체크포인트(`unet_mini_*.pth`) 보존 · `pip install pytorch-msssim optuna` (W3)

---

## 🎯 W2 탐구 과제 (제출)

위 **§6.5 심화** 셀을 바탕으로 아래 과제를 *코드 수정 + 결과 분석*과 함께 정리합니다.
단순 실행이 아니라 **"무엇을 바꿨고 / 무엇이 변했고 / 왜 그런가"** 를 반드시 적으세요.

### 과제 1 — preset & 모델 크기 (필수)
`fast`·`standard` 비교에 더해 `train_custom(base=…)` 로 모델 크기를 직접 바꿔,
파라미터 수 ↔ |Δφ|·SSIM ↔ 학습 시간의 trade-off를 표/그래프로. 학습 곡선에서 **overfitting 징후**를 찾아 설명.

### 과제 2 — Cross-domain 일반화 (필수)
4 도메인 평가 표 + 시각화. 추가로 **각 도메인의 Linear 대비 개선폭**을 계산하고,
어느 도메인에서 일반화가 깨지는지·그 원인 가설(공극률·구조·등방성)을 정량 근거와 함께 제시.

### 과제 3 — 복합 손실 함수 (선택, 도전)
§6.5-B 확장: L1 / MSE 비교에 더해 **L1 + λ·(1−SSIM) 복합 손실을 직접 구현**,
λ 를 바꿔 선명도와 |Δφ| 의 trade-off를 분석.

### 과제 4 — k 일반화 & 실패 모드 (선택, 심화)
§6.5-C/D 확장: 학습 k 와 평가 k 를 바꿔 **성능 지도**를 만들고,
per-slice 오차가 큰 슬라이스들의 **구조적 공통점**을 찾아 모델의 한계를 서술.